# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [29]:
import pandas as pd
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)
df.head()

,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


In [ ]:
#1. Create a new DataFrame that only includes customers who:
   #- have a **low total_claim_amount** (e.g., below $1,000),
   #- have a response "Yes" to the last marketing campaign.
low_claim_yes_response = df[
    (df["Total Claim Amount"] < 1000) & 
    (df["Response"] == "Yes")
]
df["Total Claim Amount"] = df["Total Claim Amount"].round()
low_claim_yes_response[["Total Claim Amount", "Response"]].reset_index(drop=True).head()


,Total Claim Amount,Response
0,484.0,Yes
1,739.0,Yes
2,547.0,Yes
3,20.0,Yes
4,60.0,Yes


In [26]:
df.dtypes

Unnamed: 0                         int64
Customer                          object
State                             object
Customer Lifetime Value          float64
Response                          object
Coverage                          object
Education                         object
Effective To Date                 object
EmploymentStatus                  object
Gender                            object
Income                             int64
Location Code                     object
Marital Status                    object
Monthly Premium Auto               int64
Months Since Last Claim          float64
Months Since Policy Inception      int64
Number of Open Complaints        float64
Number of Policies                 int64
Policy Type                       object
Policy                            object
Renew Offer Type                  object
Sales Channel                     object
Total Claim Amount               float64
Vehicle Class                     object
Vehicle Size    

In [7]:
#2. Using the original Dataframe, analyze:
   #- the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   #- compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.
yes_customers = df[df["Response"] == "Yes"]

premium_clv = (
    yes_customers
    .groupby(["Policy Type", "Gender"])[["Monthly Premium Auto", "Customer Lifetime Value"]]
    .mean()
    .round(2)
)
claims = (
    yes_customers
    .groupby(["Policy Type", "Gender"])["Total Claim Amount"]
    .mean()
    .round(2)
)

result = premium_clv.join(claims)
result


Monthly Premium Auto  Customer Lifetime Value  \
Policy Type    Gender                                                  
Corporate Auto F                      94.30                  7712.63   
               M                      92.19                  7944.47   
Personal Auto  F                      99.00                  8339.79   
               M                      91.09                  7448.38   
Special Auto   F                      92.31                  7691.58   
               M                      86.34                  8247.09   

                       Total Claim Amount  
Policy Type    Gender                      
Corporate Auto F                   433.73  
               M                   408.56  
Personal Auto  F                   452.98  
               M                   456.99  
Special Auto   F                   453.29  
               M                   429.53

In [ ]:
#3. Analyze the total number of customers who have policies in each state, 
# and then filter the results to only include states where there are more than 500 customers.

state_customers = (
    df.groupby("State")["Customer"]
    .count()
)
result = state_customers[state_customers > 500]
result
# California is leading in terms of number of customers with policies.

State
Arizona       1937
California    3552
Nevada         993
Oregon        2909
Washington     888
Name: Customer, dtype: int64

In [ ]:
#4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.
result = (
    df.groupby(["Education", "Gender"])["Customer Lifetime Value"]
      .agg(["max", "min", "median"]).round()
)

result
#The Men with High school or below educatiin level have the highest max Customer Lifetime Value.
#The Women withMaster degrees have the highest min Customer Lifetime Value.
#The customers with higher level of education (Doctor or Master) have a higher min but the lowest max Customer Lifetime Value.

max     min  median
Education            Gender                         
Bachelor             F       73226.0  1904.0  5641.0
                     M       67907.0  1898.0  5548.0
College              F       61850.0  1899.0  5624.0
                     M       61135.0  1918.0  6006.0
Doctor               F       44856.0  2396.0  5332.0
                     M       32677.0  2268.0  5578.0
High School or Below F       55277.0  2145.0  6040.0
                     M       83325.0  1941.0  6287.0
Master               F       51016.0  2418.0  5730.0
                     M       50568.0  2272.0  5579.0

In [ ]:
#5. The marketing team wants to analyze the number of policies sold by state and month. 
# Present the data in a table where the months are arranged as columns and the states are arranged as rows.
#6. Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

#*Hint:*
#- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. 
# Afterwards, you will need to sort the data by the count of policies sold in descending order.*
#- *Next, you will select the top 3 states with the highest number of policies sold.*
#- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [37]:
df["Effective To Date"] = pd.to_datetime(df["Effective To Date"])
df["Month"] = df["Effective To Date"].dt.month_name()

policies_state_month = (
    df.groupby(["State", "Month"])["Number of Policies"]
      .count()
)

policies_state_month = policies_state_month.sort_values(ascending=False)

policies_state_month

State       Month   
California  January     1918
            February    1634
Oregon      January     1565
            February    1344
Arizona     January     1008
            February     929
Nevada      January      551
Washington  January      463
Nevada      February     442
Washington  February     425
Name: Number of Policies, dtype: int64

In [40]:
top_states = (
    df.groupby("State")["Number of Policies"]
      .count()
      .sort_values(ascending=False)
      .head(3)
)

top_states

State
California    3552
Oregon        2909
Arizona       1937
Name: Number of Policies, dtype: int64

In [44]:
top_states_list = top_states.index

In [43]:
top_states_month = (
    df[df["State"].isin(top_states_list)]
    .groupby(["State", "Month"])["Number of Policies"]
    .count()
    .reset_index()
)

top_states_month

,State,Month,Number of Policies
0,Arizona,February,929
1,Arizona,January,1008
2,California,February,1634
3,California,January,1918
4,Oregon,February,1344
5,Oregon,January,1565
